# DCSASS dataset inspection
In Colab select **Runtime → Change runtime type → T4 GPU** (optional for inspection), then **Runtime → Run all**. Mount Drive when prompted. Run cells in order. Publish the repository changes first so cloning includes these utilities. No dataset statistics are assumed. Every video file is decoded once; a readable first frame does not prove the entire clip is valid.

In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO = Path('/content/AI-Crime-Investigation-FYP')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/ranaalihaider99993-source/AI-Crime-Investigation-FYP.git', str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
import torch
from scripts.common import seed_all
seed_all(42)
print('Device:', 'cuda' if torch.cuda.is_available() else 'cpu')
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT = Path('/content/drive/MyDrive/AI-Crime-Investigation-FYP')
except ImportError:
    OUTPUT = REPO / 'artifacts'
OUTPUT.mkdir(parents=True, exist_ok=True)
print('Persistent output:', OUTPUT)


In [ ]:
import kagglehub
try:
    DATA = Path(kagglehub.dataset_download('mateohervas/dcsass-dataset'))
except Exception as exc:
    raise RuntimeError('Kaggle download failed. Check network/access; if authentication is required, use Colab secrets or kagglehub.login(), never commit credentials.') from exc
print('Dataset root:', DATA)


In [ ]:
from scripts.data import inspect
import json
report = inspect(DATA, OUTPUT / 'inspection')
print(json.dumps(report, indent=2))
print('Top-level entries:', [(p.name, 'directory' if p.is_dir() else 'file') for p in sorted(DATA.iterdir())])

In [ ]:
# Inspect label file contents before declaring bounding boxes or narratives absent.
for relative in report['annotation_candidates'][:20]:
    path = DATA / relative
    print(relative, path.read_text(errors='replace')[:2000])


Review all annotation candidates in Drive/inspection/inspection.json. Record actual schemas and whether temporal labels, object boxes, and paired scenario text exist. DCSASS category/temporal labels must not be treated as bounding boxes or narrative targets. If boxes are absent, obtain a separately licensed, human-annotated Kaggle/Roboflow detection dataset before detector training. No detector is trained here. Review manifest.csv source IDs and class mapping before notebook 02; unmapped categories are excluded. Never infer ground truth narrative scenarios from class names.